In [ ]:
using Plots, Random, Statistics, LinearAlgebra, Distributions, SpecialFunctions, Optim

# Constants
c = 299792458.0
ħ = 1.054571817e-34
α = 1/137.036
qₑ = 1.602176634e-19

# Fixed parameters
ρₐ_fixed = 0.3e9 * 1.602176634e-10
Bₑ_fixed = 10.0
A_fixed = 1.0
β²_fixed = 5e4
v_lab_fixed = 242e3

function broaden_signal(signal_array, freq_array, broadening_factor=1.5)
    if broadening_factor == 1.0
        return signal_array
    end
    
    broadened = zeros(length(signal_array))
    kernel_width = broadening_factor * 0.2e6
    
    for (i, f) in enumerate(freq_array)
        if signal_array[i] > 0
            for (j, f_target) in enumerate(freq_array)
                weight = exp(-0.5 * ((f - f_target) / kernel_width)^2)
                broadened[j] += signal_array[i] * weight
            end
        end
    end
    
    if maximum(broadened) > 0
        broadened *= maximum(signal_array) / maximum(broadened)
    end
    
    return broadened
end

function axion_signal_power(ω::Float64, mₐ::Float64, Cₐᵧ::Float64, σᵥ::Float64, v_lab::Float64)
    ω_threshold = mₐ * c^2 / ħ
    if ω <= ω_threshold
        return 0.0
    end
    
    sqrt_arg = 1 - ((mₐ * c^2) / (ħ * ω))^2
    if sqrt_arg <= 0.0
        return 0.0
    end
    
    v_ω = c * sqrt(sqrt_arg)
    fₐ = mₐ * c^2 / ħ
    gₐᵧ = (α / (2π * fₐ)) * Cₐᵧ
    
    prefactor = (ρₐ_fixed / mₐ^2) * gₐᵧ^2 * Bₑ_fixed^2 * A_fixed * β²_fixed * (qₑ / ħ)
    σᵥ² = σᵥ^2
    sqrt_factor = sqrt(2 / (π * σᵥ²))/v_lab
    exp_factor = exp(-(v_ω^2 + v_lab^2) / (2 * σᵥ²))
    sinh_factor = sinh((v_ω * v_lab) / σᵥ²)
    
    result = prefactor * sqrt_factor * v_ω * exp_factor * sinh_factor
    
    if isnan(result) || isinf(result)
        return 0.0
    end
    
    return result
end

function savitzky_golay_filter(data, window_length, poly_order)
    if window_length % 2 == 0
        error("Window length must be odd")
    end
    
    if poly_order >= window_length
        error("Polynomial order must be less than window length")
    end
    
    half_window = window_length ÷ 2
    x_vals = -half_window:half_window
    A = zeros(window_length, poly_order + 1)
    
    for i in 1:window_length
        for j in 0:poly_order
            A[i, j+1] = x_vals[i]^j
        end
    end
    
    coeffs = (A' * A) \ A'
    central_coeffs = coeffs[1, :]
    
    filtered_data = copy(data)
    n = length(data)
    
    for i in (half_window + 1):(n - half_window)
        window_data = data[(i - half_window):(i + half_window)]
        filtered_data[i] = dot(central_coeffs, window_data)
    end
    
    return filtered_data
end

function generate_signal_data()
    N = 25000
    Δf = 2.e3
    f₀ = 4.218e6
    frequencies = f₀ .+ (0:N-1) * Δf
    freq_MHz = (frequencies .- f₀) / 1e6
    
    f_axion = f₀ + 10.5e6
    mₐ_true = ħ * 2π * f_axion / c^2
    Cₐᵧ_true = 1.8
    σᵥ_true = 218e3
    
    signal = broaden_signal([axion_signal_power(2π * f, mₐ_true, Cₐᵧ_true, σᵥ_true, v_lab_fixed) for f in frequencies], frequencies, 0.055)
    signal_scale_factor = 0.7*1e20
    signal .*= signal_scale_factor
    
    noise_std = 0.08
    Random.seed!(42)
    noise = noise_std * randn(N)
    
    function generate_background(f_array, f_ref, seed_val=42)
        Random.seed!(seed_val)
        r = randn(8)
        bg = zeros(length(f_array))
        
        for (i, f) in enumerate(f_array)
            f_rel = f - f_ref
            
            term1 = 1e-20 * (erf((f_rel - f_ref) / (5e6)) * (f_ref / f)^3 +
                             exp(-((f_rel - 25e6 * (1 + r[1]/15)) / (20e6 * (1 + r[2]/10)))^2))
            term2 = 4e-22 * (1 + r[3]) * sin((f_rel + r[4] * f_ref) / (2.5e6))
            term3 = 5e-24 * ((1 + r[5]) * sin((f_rel + r[6] * f_ref) / (0.25e6)) +
                             (1 + r[7]) * sin((f_rel + r[8] * f_ref) / (0.1e6)))
            
            bg[i] = abs(term1 + term2 + term3)
        end
        
        bg = bg * 1e22 * 1
        
        return bg
    end
    background = generate_background(frequencies, f₀, 42)
    
    observed = signal + noise + background
    
    sg_window_length = 221
    sg_poly_order = 4
    
    filtered_data = savitzky_golay_filter(observed, sg_window_length, sg_poly_order)
    boundary_cut = 110
    valid_indices = (boundary_cut + 1):(N - boundary_cut)
    freq_valid = frequencies[valid_indices]
    freq_MHz_valid = freq_MHz[valid_indices]
    raw_valid = observed[valid_indices]
    filtered_valid = filtered_data[valid_indices]
    residual_data = raw_valid - filtered_valid
    
    return freq_valid, freq_MHz_valid, residual_data, mₐ_true, Cₐᵧ_true, σᵥ_true, f_axion, noise_std
end

# Method 1: Direct standard deviation of residuals
function estimate_noise_direct(residual_data)
    return std(residual_data)
end

# Method 2: Three-region method from the paper
function estimate_noise_three_regions(residual_data)
    n_total = length(residual_data)
    region_size = n_total ÷ 3
    
    println("Total bins: ", n_total)
    println("Region size: ", region_size, " bins (roughly 8333 as mentioned in paper)")
    
    stds = Float64[]
    
    for i in 1:3
        start_idx = (i-1) * region_size + 1
        end_idx = (i == 3) ? n_total : i * region_size
        region_std = std(residual_data[start_idx:end_idx])
        push!(stds, region_std)
        println("Region ", i, " (bins ", start_idx, "-", end_idx, "): std = ", region_std)
    end
    
    min_std = minimum(stds)
    min_region = argmin(stds)
    println("Selected region ", min_region, " with minimum std = ", min_std)
    
    return min_std
end

function find_peak_frequency(freq_Hz, data)
    max_idx = argmax(data)
    return freq_Hz[max_idx]
end

function log_likelihood(params, freq_Hz, data, noise_std)
    f_peak, Cₐᵧ, σᵥ = params
    mₐ = ħ * 2π * f_peak / c^2
    
    model = [axion_signal_power(2π * f, mₐ, Cₐᵧ, σᵥ, v_lab_fixed) for f in freq_Hz]
    model = broaden_signal(model, freq_Hz, 0.055) .* 0.7e20
    
    residuals = data .- model
    return -0.5 * sum((residuals ./ noise_std).^2)
end

function log_prior(params, f_min, f_max)
    f_peak, Cₐᵧ, σᵥ = params
    
    if !(f_min <= f_peak <= f_max)
        return -Inf
    end
    
    if !(0.86 <= Cₐᵧ <= 41.0)
        return -Inf
    end
    
    if !(100e3 <= σᵥ <= 400e3)
        return -Inf
    end
    
    σᵥ_prior = Normal(218e3, 39e3)
    return logpdf(σᵥ_prior, σᵥ)
end

function run_mcmc(freq_Hz, data, noise_std, n_steps=20000, method_name="")
    f_min = minimum(freq_Hz)
    f_max = maximum(freq_Hz)
    
    f_peak_init = find_peak_frequency(freq_Hz, data)
    params = [f_peak_init, 1.8, 218e3]
    
    step_sizes = [5e3, 0.1, 5e3]
    
    chain = zeros(n_steps, 3)
    accepted = 0
    
    log_post_current = log_likelihood(params, freq_Hz, data, noise_std) + 
                       log_prior(params, f_min, f_max)
    
    adaptation_interval = 500
    target_acceptance = 0.25
    
    for i in 1:n_steps
        proposal = copy(params)
        proposal[1] += step_sizes[1] * randn()
        proposal[2] += step_sizes[2] * randn()
        proposal[3] += step_sizes[3] * randn()
        
        log_post_proposal = log_likelihood(proposal, freq_Hz, data, noise_std) + 
                           log_prior(proposal, f_min, f_max)
        
        α = min(1, exp(log_post_proposal - log_post_current))
        
        if rand() < α
            params = proposal
            log_post_current = log_post_proposal
            accepted += 1
        end
        
        chain[i, :] = params
        
        if i % adaptation_interval == 0 && i < n_steps/2
            current_acceptance = accepted / adaptation_interval
            if current_acceptance < target_acceptance - 0.05
                step_sizes .*= 0.8
            elseif current_acceptance > target_acceptance + 0.05
                step_sizes .*= 1.2
            end
            accepted = 0
        end
    end
    
    println(method_name, " - Final acceptance rate: ", accepted/(n_steps/2))
    return chain
end




TRUE NOISE STD (ground truth): 0.08

METHOD 1: Direct standard deviation of residuals
----------------------------------------
Estimated noise (direct): 0.08177192207537685

METHOD 2: Three-region method (from paper)
----------------------------------------
Total bins: 24780
Region size: 8260 bins (roughly 8333 as mentioned in paper)
Region 1 (bins 1-8260): std = 0.08633606282420447
Region 2 (bins 8261-16520): std = 0.07898367030060383
Region 3 (bins 16521-24780): std = 0.0798074171363026
Selected region 2 with minimum std = 0.07898367030060383
Estimated noise (3-region): 0.07898367030060383

COMPARISON:
Ground truth noise: 0.08
Direct method: 0.08177192207537685 (bias: 0.0017719220753768528)
3-region method: 0.07898367030060383 (bias: -0.0010163296993961762)

Running MCMC with Method 1 (direct)...
Method 1 - Final acceptance rate: 0.0111

Running MCMC with Method 2 (3-region)...
Method 2 - Final acceptance rate: 0.0101

FITTED PARAMETERS COMPARISON
True values:
  f_axion: 14.718 MHz
 

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://c:/Users/Lenovo/Documents/GitHub/Dark_Matter_Magnetized_Disc_Mirror_Axion_Dielectric_Haloscope_Experiment/In[5]#384:1\[90mIn[5]:384:1[0;0m]8;;\
    plot(p1, p2, p3, layout=(1,3), size=(1400, 400))
[48;2;120;70;70mend[0;0m
[90m└─┘ ── [0;0m[91minvalid identifier[0;0m